# LLM Evaluation on Kaggle

Notebook này dùng riêng cho Kaggle Notebook để benchmark các LLM trên bài toán comparative quintuple extraction.

Trước khi chạy:
1. Add project folder dưới dạng Kaggle Input Dataset hoặc clone từ GitHub.
2. Thêm `OPENROUTER_API_KEY` trong `Settings -> Secrets`.
3. Chỉnh `PROJECT_INPUT_DIR`, `DATASETS`, `SPLIT`, `MODELS`, `PROMPT_STRATEGY` ở cell cấu hình.

In [ ]:
# Cell 1 · Kaggle paths
import os

assert os.path.exists('/kaggle/working'), 'Notebook này chỉ dành cho Kaggle.'

PROJECT_INPUT_DIR = '/kaggle/input/msc-project'
WORK_DIR = '/kaggle/working/msc-project'
LLMEVAL_DIR = os.path.join(WORK_DIR, 'llm_eval')
DATASETS_ROOT = os.path.join(WORK_DIR, 'datasets')
OUTPUT_DIR = os.path.join(LLMEVAL_DIR, 'results')
CACHE_DIR = os.path.join(LLMEVAL_DIR, 'cache')

print('Input project dir:', PROJECT_INPUT_DIR)
print('Working project dir:', WORK_DIR)

In [ ]:
# Cell 2 · Prepare project in /kaggle/working
GITHUB_REPO = ''  # tùy chọn: clone nếu không dùng Kaggle Input

import os
import shutil
import subprocess

if not os.path.exists(LLMEVAL_DIR):
    if os.path.exists(PROJECT_INPUT_DIR):
        shutil.copytree(PROJECT_INPUT_DIR, WORK_DIR, dirs_exist_ok=True)
        print('Copied project from Kaggle Input Dataset.')
    elif GITHUB_REPO:
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, WORK_DIR], check=True)
        print('Cloned project from GitHub.')
    else:
        raise FileNotFoundError('Không tìm thấy project trong Kaggle Input và GITHUB_REPO đang để trống.')
else:
    print(f'Project already exists at {WORK_DIR}')

In [ ]:
# Cell 3 · Install dependencies
import os
import subprocess
import sys

reqs = os.path.join(LLMEVAL_DIR, 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', reqs, '-q'], check=True)
print('Dependencies installed.')

In [ ]:
# Cell 4 · Load OpenRouter API key from Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient

os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY not found in Kaggle Secrets.'
print('API key loaded.')

In [ ]:
# Cell 5 · Configuration
DATASETS = 'camera-coqe,vcom-data'
SPLIT = 'test'
PROMPT_STRATEGY = 'few-shot'  # zero-shot | few-shot | cot

MODELS = [
    'openai/gpt-4o-mini',
    'anthropic/claude-3.5-haiku',
    'google/gemini-2.0-flash-001',
    'deepseek/deepseek-chat',
    'qwen/qwen-2.5-72b-instruct',
    'meta-llama/llama-3.3-70b-instruct',
]

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 256
SLEEP_SECONDS = 0.3
LIMIT = 0

print('Configuration ready.')

In [ ]:
# Cell 6 · Run evaluation
import os
import subprocess
import sys

cmd = [
    sys.executable, os.path.join(LLMEVAL_DIR, 'run_eval.py'),
    '--datasets', DATASETS,
    '--split', SPLIT,
    '--models', *MODELS,
    '--prompt-strategy', PROMPT_STRATEGY,
    '--base-url', 'https://openrouter.ai/api/v1',
    '--api-key-env', 'OPENROUTER_API_KEY',
    '--temperature', str(TEMPERATURE),
    '--max-output-tokens', str(MAX_OUTPUT_TOKENS),
    '--sleep-seconds', str(SLEEP_SECONDS),
    '--datasets-root', DATASETS_ROOT,
    '--output-dir', OUTPUT_DIR,
    '--cache-dir', CACHE_DIR,
]
if LIMIT > 0:
    cmd += ['--limit', str(LIMIT)]

result = subprocess.run(cmd, text=True, capture_output=False)
print('Exit code:', result.returncode)

In [ ]:
# Cell 7 · Show summary
import json
import pathlib

summary_file = pathlib.Path(OUTPUT_DIR) / f'summary__{SPLIT}.json'

if summary_file.exists():
    with open(summary_file, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    try:
        import pandas as pd
        df = pd.DataFrame([
            {
                'dataset': r['dataset'],
                'model': r['model'],
                'E-T5-MACRO-F1': round(r.get('E-T5-MACRO-F1', 0), 4),
                'E-T4-F1': round(r.get('E-T4-F1', 0), 4),
                'E-CEE-MICRO-F1': round(r.get('E-CEE-MICRO-F1', 0), 4),
            }
            for r in rows
        ]).sort_values(['dataset', 'E-T5-MACRO-F1'], ascending=[True, False])
        print(df.to_string(index=False))
    except ImportError:
        print(rows)
else:
    print('Summary file not found.')

In [ ]:
# Cell 8 · Results location
import pathlib

results_path = pathlib.Path(OUTPUT_DIR)
print('Results saved at:', results_path)
print('Use the right-side Output/Data panel in Kaggle to download files if needed.')